# Team 3 - 전처리 & Split 노트북

팀원 C님 노트북 기반. split만 `GroupShuffleSplit`으로 변경 (팀 표준).

## 사용 방법
1. 아래 경로 변수만 자기 환경에 맞게 수정
2. 셀 순서대로 실행
3. 최종 산출물: `clean_df.csv`, `train.txt`, `val.txt`, `pill.yaml`

## 팀 Split 표준
- `GroupShuffleSplit`, groups=combo, test_size=0.2, random_state=42

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import matplotlib.patches as patches
from PIL import Image
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ===== 경로 설정 (자기 환경에 맞게 수정) =====
BASE_DIR       = r"D:\pill_detection\data"
CSV_PATH       = os.path.join(BASE_DIR, "merged_train_after_category_merge.csv")
MAPPING_PATH   = os.path.join(BASE_DIR, "final_mapping_after_category_merge.csv")
AIHUB_IMG_BASE = BASE_DIR
KAGGLE_IMG_DIR = os.path.join(BASE_DIR, "train_images")  # Kaggle train 원본 폴더
# ============================================

# 이미지 경로 찾는 함수
def get_image_path(row):
    if pd.notna(row['set_id']):
        folder_name = row['file_name'].split('_0_')[0]
        return os.path.join(AIHUB_IMG_BASE, f"TS_{int(row['set_id'])}_조합", folder_name, row['file_name'])
    else:
        return os.path.join(KAGGLE_IMG_DIR, row['file_name'])

# 데이터 로드
df = pd.read_csv(CSV_PATH, low_memory=False)
mapping = pd.read_csv(MAPPING_PATH)

print(f'CSV shape: {df.shape}')
print(f'전체 어노테이션: {len(df)}행')
print(f'전체 이미지 수: {df["file_name"].nunique()}장')
print(f'전체 클래스 수: {df["final_class_idx_merged"].nunique()}개')
print(f'\n데이터 출처:')
print(df['source'].value_counts(dropna=False))
df.head(3)

CSV shape: (40844, 45)
전체 어노테이션: 40844행
전체 이미지 수: 10697장
전체 클래스 수: 118개

데이터 출처:
source
aihub_extra    40081
NaN              763
Name: count, dtype: int64


,file_name,image_id,width,height,category_id,dl_name,dl_name_en,bbox_x,bbox_y,bbox_w,...,dl_mapping_code,dl_material,drug_dir,image_exists,json_path,light_color,set_id,source,category_id_merged,final_class_idx_merged
0,K-001900-016548-019607-029451_0_2_0_2_70_000_2...,34,976,1280,1900,보령부스파정 5mg,Buspar Tab. 5mg Boryung,644,845,189,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1899,3
1,K-001900-016548-019607-029451_0_2_0_2_75_000_2...,35,976,1280,1900,보령부스파정 5mg,Buspar Tab. 5mg Boryung,148,241,181,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1899,3
2,K-001900-016548-019607-029451_0_2_0_2_90_000_2...,36,976,1280,1900,보령부스파정 5mg,Buspar Tab. 5mg Boryung,167,248,184,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1899,3


## 팀 공통 전처리 결과 로드

이 노트북은 `team_preprocessed/clean_df.csv`를 베이스로 사용.  
(원본 merged_train → 4가지 공통 기준 제거 → clean_df.csv)

제거 기준:
1. 명백한 오류 bbox (w/h/area ≤ 0)
2. bbox 경계 초과
3. 좌표 완전 중복
4. 라벨 누락/오류 (팀원 C 로직 + 같은 클래스 중복)

결과: 40,844 → 39,864 ann / 10,697 → 10,392 img

In [3]:
# ===== 팀 공통 전처리 결과 로드 =====
CLEAN_CSV = os.path.join(BASE_DIR, "team_preprocessed", "clean_df.csv")

df_clean = pd.read_csv(CLEAN_CSV, low_memory=False)

print(f'팀 공통 clean_df 로드 완료')
print(f'Shape: {df_clean.shape}')
print(f'이미지: {df_clean["file_name"].nunique():,}장')
print(f'어노테이션: {len(df_clean):,}개')
print(f'클래스: {df_clean["final_class_idx_merged"].nunique()}개')
print(f'\n데이터 출처:')
print(df_clean['source'].value_counts(dropna=False))

# YOLO 변환에 필요한 컬럼 있는지 확인
required_cols = ['file_name', 'final_class_idx_merged', 
                 'bbox_cx_n', 'bbox_cy_n', 'bbox_w_n', 'bbox_h_n',
                 'set_id', 'source', 'dl_name']
missing = [c for c in required_cols if c not in df_clean.columns]
if missing:
    print(f'\n⚠️ 누락 컬럼: {missing}')
else:
    print(f'\n✅ YOLO 변환 필수 컬럼 모두 존재')

팀 공통 clean_df 로드 완료
Shape: (39864, 45)
이미지: 10,392장
어노테이션: 39,864개
클래스: 118개

데이터 출처:
source
aihub_extra    39131
NaN              733
Name: count, dtype: int64

✅ YOLO 변환 필수 컬럼 모두 존재


## YOLO 라벨 txt 변환

각 이미지 파일명에 해당하는 `.txt` 파일 생성.  
한 줄 = `class_idx cx_n cy_n w_n h_n` (YOLO 포맷).

출력: `D:\pill_detection\data\labels\*.txt`

In [4]:
# ===== YOLO 라벨 TXT 변환 =====
LABEL_DIR = os.path.join(BASE_DIR, "labels")
os.makedirs(LABEL_DIR, exist_ok=True)

for file_name, group in df_clean.groupby('file_name'):
    txt_name = os.path.splitext(file_name)[0] + '.txt'
    txt_path = os.path.join(LABEL_DIR, txt_name)
    
    with open(txt_path, 'w') as f:
        for _, row in group.iterrows():
            f.write(f"{int(row['final_class_idx_merged'])} "
                    f"{row['bbox_cx_n']:.6f} "
                    f"{row['bbox_cy_n']:.6f} "
                    f"{row['bbox_w_n']:.6f} "
                    f"{row['bbox_h_n']:.6f}\n")

print(f'TXT 변환 완료')
print(f'생성된 파일 수: {len(os.listdir(LABEL_DIR)):,}개')
print(f'저장 경로: {LABEL_DIR}')

# 샘플 확인 (첫 번째 파일 내용)
sample_txt = os.listdir(LABEL_DIR)[0]
sample_path = os.path.join(LABEL_DIR, sample_txt)
with open(sample_path) as f:
    print(f'\n샘플 ({sample_txt}):')
    print(f.read())

TXT 변환 완료
생성된 파일 수: 10,392개
저장 경로: D:\pill_detection\data\labels

샘플 (K-000250-000573-002483-006192_0_2_0_2_70_000_200.txt):
0 0.705943 0.252734 0.278689 0.217969
1 0.228484 0.263281 0.311475 0.232813
4 0.205943 0.710938 0.223361 0.381250
18 0.720799 0.652734 0.367828 0.389844



## Train/Val Split (팀 표준)

**기법**: `sklearn.model_selection.GroupShuffleSplit`  
**Group**: 조합 코드 (`K-AAAA-BBBB-CCCC-DDDD`)  
**비율**: train 80 / val 20  
**seed**: 42

### 기법 선택 근거
- 같은 조합(K-AAAA-BBBB-CCCC-DDDD)은 train/val에 쪼개지지 않도록 묶음
- 조합별 각도 70/75/90 이미지가 분리되는 걸 방지
- 02_eda Cell 13 시뮬레이션 결과 Combo Group 방식이 overlap 0으로 확정

```python
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(df_clean, groups=df_clean['combo']))
```

In [5]:
# ===== Train/Val Split: GroupShuffleSplit =====
from sklearn.model_selection import GroupShuffleSplit

# combo 컬럼 생성: 파일명에서 _0_ 앞부분 (조합 코드)
df_clean['combo'] = df_clean['file_name'].apply(lambda x: x.split('_0_')[0])

print(f'전체 어노테이션: {len(df_clean):,}건')
print(f'전체 이미지: {df_clean["file_name"].nunique():,}장')
print(f'전체 combo: {df_clean["combo"].nunique():,}개')
print()

# GroupShuffleSplit 적용 (combo 단위 분할)
VAL_RATIO = 0.2
SEED = 42

gss = GroupShuffleSplit(n_splits=1, test_size=VAL_RATIO, random_state=SEED)
train_idx, val_idx = next(gss.split(df_clean, groups=df_clean['combo']))

train_df = df_clean.iloc[train_idx].copy().reset_index(drop=True)
val_df   = df_clean.iloc[val_idx].copy().reset_index(drop=True)

train_files = train_df['file_name'].unique().tolist()
val_files   = val_df['file_name'].unique().tolist()

print(f'[Train]')
print(f'  어노테이션: {len(train_df):,}건')
print(f'  이미지: {len(train_files):,}장')
print(f'  combo: {train_df["combo"].nunique():,}개')
print(f'  클래스: {train_df["final_class_idx_merged"].nunique()}개')
print()
print(f'[Val]')
print(f'  어노테이션: {len(val_df):,}건')
print(f'  이미지: {len(val_files):,}장')
print(f'  combo: {val_df["combo"].nunique():,}개')
print(f'  클래스: {val_df["final_class_idx_merged"].nunique()}개')

# === 누수 검증 ===
print(f'\n=== 누수 검증 ===')
train_combos = set(train_df['combo'])
val_combos = set(val_df['combo'])
combo_overlap = train_combos & val_combos

train_imgs = set(train_files)
val_imgs = set(val_files)
img_overlap = train_imgs & val_imgs

print(f'Combo overlap: {len(combo_overlap)}건 (목표: 0)')
print(f'Image overlap: {len(img_overlap)}건 (목표: 0)')

if len(combo_overlap) == 0 and len(img_overlap) == 0:
    print('✅ 누수 없음')
else:
    print('❌ 누수 발생, split 로직 재검토 필요')

전체 어노테이션: 39,864건
전체 이미지: 10,392장
전체 combo: 3,590개

[Train]
  어노테이션: 31,844건
  이미지: 8,302장
  combo: 2,872개
  클래스: 118개

[Val]
  어노테이션: 8,020건
  이미지: 2,090장
  combo: 718개
  클래스: 118개

=== 누수 검증 ===
Combo overlap: 0건 (목표: 0)
Image overlap: 0건 (목표: 0)
✅ 누수 없음


## Kaggle 이미지 복사

Kaggle train 원본(`train_images/`)을 `kaggle_images/` 폴더로 복사.  
(팀원 C님 노트북과 동일한 폴더 구조 유지 → 팀원 혼란 방지)

220장 복사, 몇 초 걸림.

In [6]:
# ===== Kaggle 이미지 복사 =====
import shutil

KAGGLE_SRC_DIR  = os.path.join(BASE_DIR, "train_images")   # 원본
KAGGLE_COPY_DIR = os.path.join(BASE_DIR, "kaggle_images")  # 복사본
os.makedirs(KAGGLE_COPY_DIR, exist_ok=True)

# Kaggle 이미지 파일명 (source가 NaN인 것)
kaggle_files = df_clean[df_clean['set_id'].isna()]['file_name'].unique()
print(f'Kaggle 이미지: {len(kaggle_files)}개')

copied = 0
skipped = 0
missing = []
for f in kaggle_files:
    src = os.path.join(KAGGLE_SRC_DIR, f)
    dst = os.path.join(KAGGLE_COPY_DIR, f)
    
    if os.path.exists(dst):
        skipped += 1
        continue
    
    if os.path.exists(src):
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing.append(f)

print(f'복사 완료: {copied}개')
print(f'이미 있음(스킵): {skipped}개')
print(f'원본 없음: {len(missing)}개')

if missing:
    print(f'\n⚠️ 원본 없는 파일 (처음 5개):')
    for f in missing[:5]:
        print(f'  - {f}')

Kaggle 이미지: 220개
복사 완료: 220개
이미 있음(스킵): 0개
원본 없음: 0개


## train.txt / val.txt 생성

YOLO가 읽을 이미지 경로 리스트 파일.  
상대 경로 사용 (BASE_DIR 기준).

- AI Hub: `TS_{set_id}_조합/{combo}/{file_name}`
- Kaggle: `kaggle_images/{file_name}`

In [7]:
# ===== train.txt / val.txt 생성 =====
def get_relative_path(filename, df_ref):
    """file_name → BASE_DIR 기준 상대 경로"""
    row = df_ref[df_ref['file_name'] == filename].iloc[0]
    if pd.notna(row['set_id']):
        # AI Hub: TS_X_조합/조합코드/파일명
        folder_name = filename.split('_0_')[0]
        return f"TS_{int(row['set_id'])}_조합/{folder_name}/{filename}"
    else:
        # Kaggle: kaggle_images/파일명
        return f"kaggle_images/{filename}"

print('경로 생성 중...')
train_img_paths = [get_relative_path(f, df_clean) for f in train_files]
val_img_paths   = [get_relative_path(f, df_clean) for f in val_files]

# 저장
TRAIN_TXT = os.path.join(BASE_DIR, "train.txt")
VAL_TXT   = os.path.join(BASE_DIR, "val.txt")

with open(TRAIN_TXT, 'w', encoding='utf-8') as f:
    f.write('\n'.join(train_img_paths))

with open(VAL_TXT, 'w', encoding='utf-8') as f:
    f.write('\n'.join(val_img_paths))

print(f'train.txt 저장: {len(train_img_paths):,}개 → {TRAIN_TXT}')
print(f'val.txt 저장:   {len(val_img_paths):,}개 → {VAL_TXT}')

# 샘플 확인
print(f'\n=== train.txt 샘플 ===')
print('[AI Hub 샘플]')
aihub_sample = [p for p in train_img_paths if 'TS_' in p][:3]
for p in aihub_sample:
    print(f'  {p}')

print('\n[Kaggle 샘플]')
kaggle_sample = [p for p in train_img_paths if 'kaggle_images' in p][:3]
for p in kaggle_sample:
    print(f'  {p}')

경로 생성 중...
train.txt 저장: 8,302개 → D:\pill_detection\data\train.txt
val.txt 저장:   2,090개 → D:\pill_detection\data\val.txt

=== train.txt 샘플 ===
[AI Hub 샘플]
  TS_1_조합/K-000250-000573-002483-012778/K-000250-000573-002483-012778_0_2_0_2_75_000_200.png
  TS_1_조합/K-000250-000573-002483-012778/K-000250-000573-002483-012778_0_2_0_2_90_000_200.png
  TS_1_조합/K-000250-000573-002483-019552/K-000250-000573-002483-019552_0_2_0_2_70_000_200.png

[Kaggle 샘플]
  kaggle_images/K-001900-016548-019607-033009_0_2_0_2_75_000_200.png
  kaggle_images/K-001900-016548-019607-033009_0_2_0_2_90_000_200.png
  kaggle_images/K-001900-016548-021771-027926_0_2_0_2_70_000_200.png


## data.yaml 생성

YOLO 학습 설정 파일. 데이터 경로와 클래스 정보 담음.

In [8]:
# ===== data.yaml 생성 =====
import yaml

# 클래스 이름 리스트 (인덱스 순서)
class_names = (df_clean.sort_values('final_class_idx_merged')
               .groupby('final_class_idx_merged')['dl_name']
               .first()
               .tolist())

data = {
    'path': BASE_DIR,
    'train': 'train.txt',
    'val': 'val.txt',
    'nc': len(class_names),
    'names': class_names
}

YAML_PATH = os.path.join(BASE_DIR, "pill.yaml")
with open(YAML_PATH, 'w', encoding='utf-8') as f:
    yaml.dump(data, f, allow_unicode=True, default_flow_style=False)

print(f'yaml 생성 완료: {YAML_PATH}')
print(f'클래스 수: {len(class_names)}')
print(f'첫 5개 클래스:')
for i, name in enumerate(class_names[:5]):
    print(f'  [{i}] {name}')
print(f'...')
print(f'  [{len(class_names)-1}] {class_names[-1]}')

yaml 생성 완료: D:\pill_detection\data\pill.yaml
클래스 수: 118
첫 5개 클래스:
  [0] 마그밀정(수산화마그네슘)
  [1] 게보린정 300mg/PTP
  [2] 알마겔정(알마게이트)(수출명:유한가스트라겔정)
  [3] 보령부스파정 5mg
  [4] 뮤테란캡슐 100mg
...
  [117] 디카테오엘씨정 10mg/병


## YOLO 표준 구조로 재배치

YOLO 학습을 위해 `dataset_yolo/` 폴더에 표준 구조로 이미지/라벨 복사.

```
dataset_yolo/
├── images/
│   ├── train/   ← 8,302장
│   └── val/     ← 2,090장
├── labels/
│   ├── train/   ← 8,302개 txt
│   └── val/     ← 2,090개 txt
└── pill.yaml    ← 학습용 yaml (재배치된 경로)
```

train.txt의 이미지 경로를 따라 `BASE_DIR`에서 이미지를 찾아 복사하고,  
같은 이름의 라벨을 `labels/` 평면 폴더에서 찾아 복사.

In [9]:
# ===== dataset_yolo 폴더 재배치 =====
import shutil
from pathlib import Path

BASE = Path(BASE_DIR)
OUT = BASE / "dataset_yolo"

TRAIN_TXT = BASE / "train.txt"
VAL_TXT = BASE / "val.txt"
LABELS_DIR = BASE / "labels"
YAML_SRC = BASE / "pill.yaml"

# 출력 폴더 생성
for p in [
    OUT / "images" / "train",
    OUT / "images" / "val",
    OUT / "labels" / "train",
    OUT / "labels" / "val",
]:
    p.mkdir(parents=True, exist_ok=True)

def copy_split(split_txt: Path, split_name: str):
    with open(split_txt, "r", encoding="utf-8") as f:
        rel_paths = [line.strip() for line in f if line.strip()]

    copied_img = 0
    copied_lbl = 0
    missing_img = []
    missing_lbl = []

    for rel in rel_paths:
        img_src = BASE / rel
        stem = Path(rel).stem
        lbl_src = LABELS_DIR / f"{stem}.txt"

        img_dst = OUT / "images" / split_name / img_src.name
        lbl_dst = OUT / "labels" / split_name / f"{stem}.txt"

        if img_src.exists():
            shutil.copy2(img_src, img_dst)
            copied_img += 1
        else:
            missing_img.append(str(img_src))
            continue

        if lbl_src.exists():
            shutil.copy2(lbl_src, lbl_dst)
            copied_lbl += 1
        else:
            missing_lbl.append(str(lbl_src))

    print(f"\n[{split_name}]")
    print(f"  총 대상 이미지 : {len(rel_paths):,}")
    print(f"  복사된 이미지  : {copied_img:,}")
    print(f"  복사된 라벨    : {copied_lbl:,}")
    print(f"  누락 이미지    : {len(missing_img):,}")
    print(f"  누락 라벨      : {len(missing_lbl):,}")

    if missing_img:
        print("\n  이미지 누락 예시")
        for x in missing_img[:5]:
            print("   ", x)

    if missing_lbl:
        print("\n  라벨 누락 예시")
        for x in missing_lbl[:5]:
            print("   ", x)

# train / val 복사
copy_split(TRAIN_TXT, "train")
copy_split(VAL_TXT, "val")

# 기존 yaml 읽어서 names, nc 유지
with open(YAML_SRC, "r", encoding="utf-8") as f:
    y = yaml.safe_load(f)

new_yaml = {
    "path": str(OUT).replace("\\", "/"),
    "train": "images/train",
    "val": "images/val",
    "nc": y["nc"],
    "names": y["names"],
}

yaml_out = OUT / "pill.yaml"
with open(yaml_out, "w", encoding="utf-8") as f:
    yaml.dump(new_yaml, f, allow_unicode=True, sort_keys=False)

print("\n새 YAML 저장 완료")
print(yaml_out)
print("\n완료된 구조:")
print(OUT / "images" / "train")
print(OUT / "images" / "val")
print(OUT / "labels" / "train")
print(OUT / "labels" / "val")


[train]
  총 대상 이미지 : 8,302
  복사된 이미지  : 8,302
  복사된 라벨    : 8,302
  누락 이미지    : 0
  누락 라벨      : 0

[val]
  총 대상 이미지 : 2,090
  복사된 이미지  : 2,090
  복사된 라벨    : 2,090
  누락 이미지    : 0
  누락 라벨      : 0

새 YAML 저장 완료
D:\pill_detection\data\dataset_yolo\pill.yaml

완료된 구조:
D:\pill_detection\data\dataset_yolo\images\train
D:\pill_detection\data\dataset_yolo\images\val
D:\pill_detection\data\dataset_yolo\labels\train
D:\pill_detection\data\dataset_yolo\labels\val


In [10]:
# ===== 재배치 검증 =====
OUT = Path(BASE_DIR) / "dataset_yolo"

for split in ["train", "val"]:
    img_dir = OUT / "images" / split
    lbl_dir = OUT / "labels" / split

    imgs = sorted(img_dir.glob("*.*"))
    lbls = sorted(lbl_dir.glob("*.txt"))

    print(f"\n[{split}]")
    print(f"이미지 수: {len(imgs):,}")
    print(f"라벨 수  : {len(lbls):,}")

    img_stems = {p.stem for p in imgs}
    lbl_stems = {p.stem for p in lbls}

    print(f"이미지만 있음: {len(img_stems - lbl_stems):,}")
    print(f"라벨만 있음  : {len(lbl_stems - img_stems):,}")


[train]
이미지 수: 8,302
라벨 수  : 8,302
이미지만 있음: 0
라벨만 있음  : 0

[val]
이미지 수: 2,090
라벨 수  : 2,090
이미지만 있음: 0
라벨만 있음  : 0
